# R18-H194 - the instrument adjudication: value comparator vs NLI vs fuzzy on a balanced blind bench

**Round R18 - executor batch 2026-07-07.** The H172 instrument decision was reopened by an accidental independent
replication: run A (unbalanced audit) scored NLI 94%/CONFIRMED, run B (balanced 25/25) 78%/REFUTED with catalogued
value-identity failures (present exact numerics scored low, absent look-alikes scored high). Both runs independently
named the same successor - a **deterministic value-extraction comparator**. This notebook adjudicates the three
candidate scorers on a **pre-registered, balanced, blind-labelled** 60-pair bench.

**Protocol (order is binding):** the bench is constructed and adjudicated **BLIND** (labels assigned by reading each
render/gold pair) and **LOCKED to disk BEFORE any scorer runs**. The bench section below is clearly separated from the
scorer section. Sources: the 33-gold small set, run B's disagreement material, and fresh renders from the read-only
neo4j2 graph (hard absences: same attribute on the wrong device, near-miss numerics, sibling look-alikes).

**Acceptance bar (registered):** comparator >= 90% overall AND wins the numeric strata; NLI >= 85% on the prose-fact
stratum determines its residual role; the shipped instrument is a router (comparator for unit/numeric/dimension golds,
NLI-or-word-overlap for prose/feature golds), which must beat any single scorer.

Graph: **read-only neo4j2** (`bolt://user-konrad.jelen-kgf-neo4j2:7687`). GPU 2 (RTX 5000 Ada, sm_89), fp16 eager
(DebertaV2 has no SDPA path in transformers 5.13); CPU fallback on OOM.

In [ ]:
# GPU selection FIRST - set CUDA env before importing torch (nvidia-smi index 2 = RTX 5000 Ada, sm_89)
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="2"
os.environ["HF_HUB_OFFLINE"]="1"                      # model is cached; no network
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rich import print as rprint

NLI_MODEL="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"   # 3-label NLI cross-encoder, cached
nli_tok=AutoTokenizer.from_pretrained(NLI_MODEL)
DEV="cuda"
try:
    nli_model=AutoModelForSequenceClassification.from_pretrained(
        NLI_MODEL, dtype=torch.float16, attn_implementation="eager").to("cuda").eval()
except RuntimeError as e:                             # OOM: GPU 2 shared with another executor -> CPU
    rprint(f"[yellow]GPU load failed ({str(e)[:60]}...) - falling back to CPU[/yellow]")
    DEV="cpu"; nli_model=AutoModelForSequenceClassification.from_pretrained(
        NLI_MODEL, dtype=torch.float32, attn_implementation="eager").to("cpu").eval()
ENT_IDX=[i for i,l in nli_model.config.id2label.items() if l=="entailment"][0]
rprint(f"[green]NLI[/green] {NLI_MODEL} | entailment idx={ENT_IDX} | {next(nli_model.parameters()).dtype} on {DEV}")

In [ ]:
# Imports + read-only graph connection
import re, json, math, pickle, hashlib, time, datetime, unicodedata, itertools
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, yaml
from rich.progress import Progress

os.environ["NEO4J_URI"]="bolt://user-konrad.jelen-kgf-neo4j2:7687"   # READ-ONLY
os.environ["NEO4J_USER"]="neo4j"; os.environ["NEO4J_PASSWORD"]="kgfoundry"
np.random.seed(42)
from knowledge_graph_foundry import load_settings, Foundry
from knowledge_graph_foundry.graph.graphrag import vector_query
from neo4j import GraphDatabase
settings=load_settings(Path("../config.yml"))
TOP_K=settings.graphrag.top_k; VEC_INDEX=settings.graphrag.vector_index_name; REL_LIMIT=15
rprint(f"[cyan]config[/cyan] top_k={TOP_K} vec_index={VEC_INDEX}")

In [ ]:
# Deterministic fuzzy-evidence matcher - VERBATIM from the H34/H61/H81 harness (the incumbent under test)
def _norm(s): return re.sub(r"\s+"," ",(s or "").casefold())
def value_tokens(t): return re.findall(r"[\w.\-/]*\d[\w.\-/]*", t)
_UNIT=r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"
def fuzzy_present(gold, ctx):
    ng=_norm(gold)
    if ng in ctx: return True
    sq=re.sub(r"[\s,()]","",ctx); sk=re.sub(r"[\s,()]","",re.sub(_UNIT,"",ng))
    if any(c.isdigit() for c in sk) and len(sk)>=5 and sk in sq: return True
    tok=value_tokens(gold)
    if tok:
        hit=sum(1 for t in tok if _norm(t) in ctx or re.sub(r"[\s,()]","",_norm(t)) in sq)
        return hit>=max(1,len(tok)//2+(len(tok)%2))
    w=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ng)); cw=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ctx))
    return bool(w) and len(w&cw)/len(w)>=0.6

In [ ]:
# Graph pull (read-only) + render primitives - reused from potentials_r10 / token_economy_r19
d=GraphDatabase.driver(os.environ["NEO4J_URI"], auth=("neo4j","kgfoundry"))
with d.session() as s:
    ents=s.run("MATCH (e:Entity) RETURN e.id AS id,e.name AS name,e.description AS description,"
               "properties(e) AS props,labels(e) AS types").data()
    edges=s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                "RETURN DISTINCT a.id AS a,b.id AS b,type(r) AS rel").data()
    prop_rows=s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) RETURN e.id AS eid,p.text AS text").data()
    alias_rows=s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id "
                     "RETURN e.id AS eid,collect(DISTINCT a.id)[..5] AS aliases").data()
d.close()
node={r["id"]:r for r in ents}; names={r["id"]:r["name"] for r in ents}
prim_type={r["id"]:(r["types"][-1] if r["types"] else "Entity") for r in ents}
props_by=defaultdict(list); [props_by[r["eid"]].append(r["text"]) for r in prop_rows]
alias_by={r["eid"]:r["aliases"] for r in alias_rows}
adj=defaultdict(set); rels_by=defaultdict(list)
for e in edges:
    adj[e["a"]].add(e["b"]); adj[e["b"]].add(e["a"])
    rels_by[e["a"]].append((e["rel"],e["b"])); rels_by[e["b"]].append((e["rel"],e["a"]))
def spec_of(r): return {k.removeprefix("prop_"):v for k,v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r=node[nid]; spec=dict(spec_of(r))
    for a in [a for a in alias_by.get(nid,[]) if a in node]:
        for k,v in spec_of(node[a]).items(): spec.setdefault(k,v)
    return spec
def base_render(nid):
    r=node[nid]; spec=merged_spec(nid); al=[a for a in alias_by.get(nid,[]) if a in node]
    aka=(f"Also known as: {', '.join(names.get(a,'') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid): return base_render(nid)+" "+" ; ".join(f"{t} -> {names.get(b,'')}" for t,b in rels_by.get(nid,[])[:REL_LIMIT])
def units_of(ids): return [seed_render(n) for n in ids] + [t for n in ids for t in props_by.get(n,[])]
rprint(f"[green]pulled[/green] entities {len(ents)} edges {len(edges)} propositions {len(prop_rows)}")

In [ ]:
# Probes + cached question embeddings -> per-probe retrieved seeds (production vec-8 path, deterministic)
probes=yaml.safe_load(Path("../tests/probes/cpap-probe-set.yml").read_text())
gold_probes=[p for p in probes if p.get("gold_evidence")]
golds=[(p["id"],g) for p in gold_probes for g in p["gold_evidence"]]
QCACHE=Path(".token_economy_r19_qcache.pkl")
qcache=pickle.load(open(QCACHE,"rb"))
def qkey(q): return hashlib.md5(q.encode()).hexdigest()
seeds_by={}
with Foundry(settings) as f:
    for p in gold_probes:
        v=qcache[qkey(p["question"])]
        seeds_by[p["id"]]=[x["id"] for x in vector_query(f.driver,v,VEC_INDEX,top_k=TOP_K) if x["id"] in node]
rprint(f"[green]retrieved seeds[/green] for {len(gold_probes)} probes | {len(golds)} golds")

## Bench construction - BLIND adjudication, LOCKED before any scorer runs

The 60 pairs are assembled from two sources: (1) the deterministic 50-pair audit design over the 33-gold set (33
gold-vs-own-context + 17 gold-vs-foreign-context, the run-B construction), and (2) 10 fresh pairs built from clean
single-product renders in neo4j2 to strengthen the dimension-triple, feature-name and prose strata and to inject hard
absences. **Every label below was adjudicated by the executor by reading the rendered context against the gold** -
verifying literal value presence and correct attribution in the current graph. Labels are locked to
`data/processed/instrument-bench-h194.json` in the next cell, before the scorer section.

Three labels **diverge from run B** (#9, #19, #21): the graph was re-embedded since run B (`chore: update embeddings`),
and the current retrieval surfaces the spec-bearing entities run B's seeds missed - the SleepStyle/HC230 dimensions and
the DreamStation 1.98 kg weight are genuinely present now. Independent adjudication follows the current graph.

In [ ]:
# Reconstruct the 50 audit pairs (deterministic) + 10 clean-render additions; attach BLIND executor labels
def strat_of(g):
    gg=_norm(g)
    if re.search(r"\d+\s*[x*]\s*\d+\s*[x*]\s*\d+",gg) or ("mm" in gg and len(re.findall(r"\d+",gg))>=3): return "dimension"
    if any(w in gg for w in ["autoset for her","sleep onset detection"]): return "feature"
    if not re.search(r"\d", gg): return "prose"
    return "numeric"

# 50-pair construction (verbatim run-B deterministic mapping)
raw=[]
for pid,g in golds: raw.append(dict(kind="pos", gold=g, src=pid, ctx_ids=list(seeds_by[pid]), src_probe=pid))
_idxs=list(range(0,len(golds),2))[:17]; _others=[p["id"] for p in gold_probes]
for i in _idxs:
    pid,g=golds[i]; cand=[o for o in _others if o!=pid]; cp=cand[(i*7)%len(cand)]
    raw.append(dict(kind="neg", gold=g, src=pid, ctx_ids=list(seeds_by[cp]), src_probe=cp))
assert len(raw)==50

# BLIND labels (1 = gold fact present/attributable in the rendered context, 0 = absent), from reading current renders
ADJ50={0:0,1:0,2:1,3:1,4:1,5:1,6:1,7:1,8:1,9:1,10:1,11:1,12:0,13:0,14:1,15:0,16:1,17:1,18:1,19:1,
 20:0,21:1,22:1,23:0,24:1,25:1,26:0,27:1,28:1,29:1,30:0,31:1,32:1,33:0,34:1,35:0,36:0,37:0,38:1,39:0,
 40:0,41:0,42:0,43:0,44:0,45:0,46:0,47:1,48:0,49:1}
RAT50={
 0:"AirSense 11 40oz/1130g weight not retrieved into P01 context",
 1:"'40 oz' weight spec absent from P01 seeds",
 2:"Pressure Setting range=4-20 cmH2O present (DreamStation)",
 3:"iBreeze render states '<28 dB at 10 cmH2O' - 28 dB(A) supported",
 4:"DreamStation warranty=2 (years) present",
 5:"Adjustable Delay/Ramp ramp_period 0-45 present",
 6:"Water Capacity=380 mL present",
 7:"AirStart 10 operating_altitude_max=2591 m present",
 8:"Power consumption typical=9.0W present",
 9:"HC230/SleepStyle dimensions_mm '275 x 170 x 140' present (DIVERGES from run B - current graph retrieves it)",
 10:"DreamStation data_storage_sd_card='> 1 year' present",
 11:"AirStart 10 weight=1106 g present",
 12:"AirSense 11 1130 g not retrieved into P11 context",
 13:"no 28 dB in P12 (prisma 26 dB, RESmart <30) - iBreeze not retrieved",
 14:"prisma SOFT plus sound_pressure_level_db=26 present",
 15:"AirSense 11 3010 m altitude not retrieved into P13",
 16:"AirStart 10 altitude 2591 m present",
 17:"AirSense 11 humidifier_capacity_ml=380 present",
 18:"iBreeze water_capacity_ml=290 present",
 19:"DreamStation weight_with_humidifier=1.98 kg present (DIVERGES from run B)",
 20:"only RESmart manual_version V2.4 present - no 2.4 kg weight",
 21:"SleepStyle dimensions_mm '275 x 170 x 140' present in P16 (DIVERGES from run B)",
 22:"iBreeze dimension 238/178/128 mm present",
 23:"iBreeze 0-60 min ramp not retrieved; P17 shows DreamStation 0-45 only",
 24:"DreamStation smart_ramp_time '0 to 45' present",
 25:"AirStart 10 sound_pressure_level=26.6 present",
 26:"27 dBA (AirSense 11) not present in P18",
 27:"'SmartRamp maintains a constant lower pressure' present verbatim",
 28:"supports_mode -> AutoSet for Her present",
 29:"'designed to help patients gradually acclimate' present",
 30:"context has 'pressure oscillations' but not 'amplitude'",
 31:"'AutoRamp with sleep onset detection' present",
 32:"'EPR reduces the pressure during expiration' present verbatim",
 33:"1130 g absent from DreamStation P02 context",
 34:"iBreeze pressure range 4-20 cmH2O present (value valid in this context)",
 35:"no warranty/2-year statement in AirStart P07 context (bare '2' unrelated)",
 36:"380 mL absent from P21 (EZ-Start) context",
 37:"9.0W absent from P12 context",
 38:"P02 is DreamStation - shares data_storage_sd_card '> 1 year'; present",
 39:"1130 g absent from P17 context",
 40:"AirStart sound is 26.6 not 26 dB - near-miss absent",
 41:"2591 m absent from Seattle-PAP P22 context",
 42:"290 ml absent from P12 context",
 43:"2.4 kg absent from iBreeze P03 context",
 44:"238*178*128 absent from P18 context",
 45:"0-45 min ramp absent from P08 power context",
 46:"27 dBA absent from P23 context",
 47:"P13 AirSense context has supports_mode -> AutoSet for Her; present",
 48:"amplitude of oscillations absent from P04 context",
 49:"P18 has 'expiratory pressure relief'/EPR - entails reducing pressure during expiration"}

# 10 clean-render additions (product context = product entity + its spec/accessory/feature 1-hop neighbours)
def clean_ids(name):
    nid=None
    if name in {r["name"] for r in ents}:
        nid=[i for i,n in names.items() if n==name][0]
    else:
        nid=[i for i,n in names.items() if name.lower() in (n or "").lower()][0]
    ids=[nid]+[b for b in sorted(adj[nid])[:25]
               if any(t in ("Specification","Accessory","ComfortFeature","OperatingMode") for t in node[b]["types"])]
    return ids
ADD=[("238*178*128 mm","iBreeze CPAP System",1,"iBreeze own dimension 238/178/128 mm present"),
 ("3,010 m","AirSense 11",1,"AirSense 11 operating_altitude_max=3010 m present"),
 ("275mm x 170mm x 140mm","iBreeze CPAP System",0,"iBreeze dims 238/178/128 not SleepStyle 275/170/140 - sibling look-alike"),
 ("1130 g","AirStart 10 CPAP",0,"1130 g is AirSense 11 weight not AirStart 10"),
 ("3,010 m","AirStart 10 CPAP",0,"AirStart altitude 2591 m not 3010 - same-attribute near-miss (fuzzy FP)"),
 ("26 dB(A)","AirSense 11",0,"26 dB(A) is prisma's not AirSense 11's"),
 ("AutoSet for Her","DreamStation CPAP",0,"DreamStation lacks AutoSet for Her - absent (fuzzy fires on 'for'/'her')"),
 ("reduces the pressure during expiration","AirStart 10 CPAP",0,"AirStart 10 has no expiratory pressure relief - topical but false"),
 ("380 mL","DreamStation CPAP",0,"380 mL is AirSense 11 humidifier not DreamStation"),
 ("1106 g","AirSense 11",0,"1106 g is AirStart 10 weight not AirSense 11 - sibling look-alike")]

BENCH=[]
for i,r in enumerate(raw):
    BENCH.append(dict(i=i, gold=r["gold"], kind=r["kind"], ctx_ids=r["ctx_ids"], src_probe=r["src_probe"],
                      stratum=strat_of(r["gold"]), label=ADJ50[i], rationale=RAT50[i], source="audit-50"))
for k,(gold,prod,lab,rat) in enumerate(ADD):
    BENCH.append(dict(i=50+k, gold=gold, kind="clean", ctx_ids=clean_ids(prod), src_probe=None,
                      product=prod, stratum=strat_of(gold), label=lab, rationale=rat, source="clean-render"))
rprint(f"[green]bench assembled[/green] {len(BENCH)} pairs")

In [ ]:
# Blind-adjudication table + composition assertions (bench is fixed here, before any scorer)
present=sum(b["label"] for b in BENCH); absent=len(BENCH)-present
strata=Counter(b["stratum"] for b in BENCH); sp=Counter((b["stratum"],b["label"]) for b in BENCH)
assert len(BENCH)==60, "expect 60 pairs"
assert present==30 and absent==30, f"expect 30/30, got {present}/{absent}"
print(f"{'#':>3} {'strat':10} {'lbl':3} {'src':11} gold")
for b in BENCH:
    print(f"{b['i']:>3} {b['stratum']:10} {b['label']:^3} {b.get('src_probe') or b.get('product',''):11.11} {b['gold'][:40]}")
rprint(f"\n[bold cyan]bench[/bold cyan] n=60 present={present} absent={absent}")
for s in ["numeric","dimension","feature","prose"]:
    rprint(f"  {s:10} total {strata[s]:2}  present {sp[(s,1)]}  absent {sp[(s,0)]}")

In [ ]:
# LOCK the bench to disk BEFORE the scorer section (pairs + blind labels + adjudication)
Path("../data/processed").mkdir(parents=True, exist_ok=True)
bench_out=dict(
  round="R18", hypothesis="H194", utc=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
  graph="neo4j2", design="60 pairs, 30 present / 30 absent, stratified; blind-adjudicated before scorers",
  strata={s:{"total":strata[s],"present":sp[(s,1)],"absent":sp[(s,0)]} for s in ["numeric","dimension","feature","prose"]},
  divergence_from_runB=[9,19,21],
  pairs=[dict(i=b["i"], gold=b["gold"], stratum=b["stratum"], label=b["label"], kind=b["kind"],
              source=b["source"], src_probe=b.get("src_probe"), product=b.get("product"),
              ctx_ids=b["ctx_ids"], rationale=b["rationale"]) for b in BENCH])
LOCK=Path("../data/processed/instrument-bench-h194.json"); LOCK.write_text(json.dumps(bench_out,indent=2,default=str))
rprint(f"[bold green]BENCH LOCKED[/bold green] -> {LOCK} ({len(BENCH)} pairs, labels frozen)")

## Scorers - run AFTER the bench is locked

Four scorers evaluated against the frozen blind labels: the incumbent **fuzzy matcher**, the **NLI** entailer (mDeBERTa,
atomic-unit max-entailment, thresholds 0.5 and 0.8), the **deterministic value comparator** built here (unit-aware
numeric normalization, key-scoped attribution, glyph/whitespace normalization per the H190 operator, dimension-triple
handling in any separator/order), and a lightweight **word-overlap** lexical scorer for the prose/feature strata. The
**router** ships value-comparator decisions on unit/numeric/dimension golds and best-of-NLI/word-overlap on
prose/feature golds.

In [ ]:
# NLI sufficiency scorer: max entailment of the gold over windowed atomic context units
@torch.inference_mode()
def nli_entail(gold, units, bs=48):
    hl=len(nli_tok(gold, add_special_tokens=False)["input_ids"]); L=512-hl-3; step=max(32,L-40)
    prem=[]
    for u in units:
        pids=nli_tok(u, add_special_tokens=False)["input_ids"]
        wins=[pids] if len(pids)<=L else [pids[i:i+L] for i in range(0,len(pids),step)]
        for w in wins: prem.append(nli_tok.decode(w))
    if not prem: return 0.0
    best=0.0
    for i in range(0,len(prem),bs):
        b=prem[i:i+bs]
        enc=nli_tok(b,[gold]*len(b),return_tensors="pt",truncation="only_first",padding=True,max_length=512).to(DEV)
        best=max(best, torch.softmax(nli_model(**enc).logits.float(),-1)[:,ENT_IDX].max().item())
    return best
nli_entail("warm up premise", ["warm up hypothesis"])
nli_scores={}
with Progress() as prog:
    t=prog.add_task("[cyan]NLI scoring 60 pairs", total=len(BENCH))
    for b in BENCH:
        nli_scores[b["i"]]=nli_entail(b["gold"], units_of(b["ctx_ids"])); prog.advance(t)
rprint("[green]NLI scored[/green] 60 pairs")

In [ ]:
# DETERMINISTIC VALUE-EXTRACTION COMPARATOR (built for H194)
# glyph/whitespace normalization (H190 operator: strip trademark glyphs -> NFKC -> fold nbsp/fullwidth -> lower/collapse)
_TM=dict.fromkeys(map(ord,"\u00ae\u2122\u00a9"),None)
def gnorm(s):
    s=(s or "").translate(_TM); s=unicodedata.normalize("NFKC",s)
    s=s.replace("\u00a0"," ").replace("\u00d7","x").replace("*","x").replace("\u00b7","x")
    s=re.sub(r"(?<=\d),(?=\d)","",s)                                   # strip digit-grouping commas
    return re.sub(r"\s+"," ",s.casefold()).strip()
UNITWORD={"mm":"len_mm","cm":"len_cm","g":"mass_g","kg":"mass_kg","oz":"mass_oz","ml":"vol_ml","l":"vol_l",
   "db":"sound_db","dba":"sound_db","w":"power_w","hz":"freq_hz","cmh2o":"press","m":"alt_m",
   "min":"time_min","mins":"time_min","minute":"time_min","minutes":"time_min","year":"warr_y","years":"warr_y"}
FAM_EQ={"len_mm":{"len_mm"},"len_cm":{"len_cm"},"alt_m":{"alt_m"},"mass_g":{"mass_g"},"mass_kg":{"mass_kg"},
   "mass_oz":{"mass_oz"},"vol_ml":{"vol_ml","vol_l"},"sound_db":{"sound_db"},"power_w":{"power_w"},
   "time_min":{"time_min"},"warr_y":{"warr_y"},"press":{"press"},"freq_hz":{"freq_hz"}}
def key_family(k):
    k=k.lower()
    if "dimension" in k or re.search(r"_mm\b",k) or "length_mm" in k: return "len_mm"
    if "altitude" in k: return "alt_m"
    if k.endswith("_kg") or "weight_kg" in k: return "mass_kg"
    if re.search(r"_g\b",k): return "mass_g"
    if "_oz" in k: return "mass_oz"
    if re.search(r"_ml\b",k) or "capacity_ml" in k or "water" in k: return "vol_ml"
    if "sound" in k or re.search(r"_db\b",k) or "noise" in k: return "sound_db"
    if "power" in k or "consumption" in k: return "power_w"
    if "ramp" in k or "delay" in k: return "time_min"
    if "warranty" in k: return "warr_y"
    if "pressure" in k: return "press"
    return None
def nums_in(v): return re.findall(r"\d+(?:\.\d+)?", str(v).replace(",",""))
def ctx_quantities(ids):
    """(number, unit-family) multiset attributed from the render's entity specs + prose - key/context scoped."""
    Q=set()
    for nid in ids:
        spec=merged_spec(nid); unit_for={}
        for k,v in spec.items():
            if k.endswith("_unit"):
                fam=UNITWORD.get(gnorm(str(v)).replace(" ",""))
                if fam: unit_for[k[:-5]]=fam
        for k,v in spec.items():
            fam=key_family(k) or unit_for.get(k)
            if fam:
                for n in nums_in(v): Q.add((n,fam))
        text=gnorm(seed_render(nid)+" "+" ".join(props_by.get(nid,[])))
        for m in re.finditer(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", text):
            fam=UNITWORD.get(m.group(2).replace("(a)",""))
            if fam: Q.add((m.group(1),fam))
    return Q
def parse_gold(gold):
    g=gnorm(gold)
    if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g): return ("dim", re.findall(r"\d+(?:\.\d+)?", g))
    if "sd card" in g: return ("sdcard", None)
    m=re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|cm h2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", g)
    rng=re.search(r"(\d+(?:\.\d+)?)\s*(?:to|-)\s*(\d+(?:\.\d+)?)", g)
    if m:
        u=m.group(2).replace("(a)","").replace("cm h2o","cmh2o").replace(" ",""); fam=UNITWORD.get(u)
        if rng and rng.group(2): return ("range",(rng.group(1),rng.group(2),fam))
        return ("num",(m.group(1),fam))
    if rng and rng.group(2):
        fam="press" if "cmh2o" in g or "cm h2o" in g else ("time_min" if "min" in g else None)
        return ("range",(rng.group(1),rng.group(2),fam))
    return ("other", None)
def comparator(gold, ids):
    """returns True/False for numeric/dimension/sdcard golds; None (abstain) for prose/feature."""
    kind,payload=parse_gold(gold); Q=ctx_quantities(ids)
    T=gnorm(" ".join(seed_render(n)+" "+" ".join(props_by.get(n,[])) for n in ids))
    if kind=="dim":
        a,b,c=payload; mm={n for n,f in Q if f=="len_mm"}
        if {a,b,c}<=mm: return True
        t=T.replace(" ","")
        return any(re.search(r"(?<!\d)"+p[0]+"x"+p[1]+"x"+p[2]+r"(?!\d)", t) for p in itertools.permutations([a,b,c]))
    if kind=="num":
        n,fam=payload
        if fam is None: return any(x==n for x,_ in Q)
        eq=FAM_EQ.get(fam,{fam}); return any(x==n and f in eq for x,f in Q)
    if kind=="range":
        a,b,fam=payload; t=T.replace(" ","")
        if re.search(r"(?<!\d)"+a+r"\s*-\s*"+b,T) or (a+"-"+b) in t or (a+"to"+b) in t: return True
        if fam:
            eq=FAM_EQ.get(fam,{fam}); xs={x for x,f in Q if f in eq}; return a in xs and b in xs
        return False
    if kind=="sdcard":
        t=T.replace(" ",""); return ("sdcard" in t) and (">1year" in t or "1year" in t)
    return None                                                         # prose/feature -> abstain, router uses NLI/overlap
# word-overlap lexical scorer for prose/feature (deterministic)
def word_overlap(gold, ids, thr=0.6):
    ng=gnorm(gold); ctx=gnorm(" ".join(units_of(ids)))
    w=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ng)); cw=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ctx))
    return bool(w) and len(w&cw)/len(w)>=thr
rprint("[green]comparator + word-overlap ready[/green]")

In [ ]:
# Score every scorer against the frozen blind labels; router = comparator(numeric/dim) + best-of(NLI,overlap)(prose/feat)
NUMERIC_STRATA={"numeric","dimension"}
def agree(preds):
    return {s:(np.mean([preds[b["i"]]==b["label"] for b in BENCH if b["stratum"]==s]) if any(b["stratum"]==s for b in BENCH) else None)
            for s in ["numeric","dimension","feature","prose"]} | {"overall":np.mean([preds[b["i"]]==b["label"] for b in BENCH])}
fuzzy_pred={b["i"]:int(fuzzy_present(b["gold"], _norm(" ".join(units_of(b["ctx_ids"]))))) for b in BENCH}
nli05={b["i"]:int(nli_scores[b["i"]]>=0.5) for b in BENCH}
nli08={b["i"]:int(nli_scores[b["i"]]>=0.8) for b in BENCH}
comp_pred={}; comp_abstain=set()
for b in BENCH:
    r=comparator(b["gold"], b["ctx_ids"])
    if r is None: comp_abstain.add(b["i"]); comp_pred[b["i"]]=0          # abstain scored as absent for standalone view
    else: comp_pred[b["i"]]=int(r)
overlap_pred={b["i"]:int(word_overlap(b["gold"], b["ctx_ids"])) for b in BENCH}
# per-stratum best-of: for each non-numeric stratum pick the better standalone scorer (NLI@0.5 vs word-overlap)
def strat_acc(pred,s): return np.mean([pred[b["i"]]==b["label"] for b in BENCH if b["stratum"]==s])
best_nonnum={s:("nli@0.5" if strat_acc(nli05,s)>=strat_acc(overlap_pred,s) else "word_overlap")
             for s in ["feature","prose"]}
PICK={"nli@0.5":nli05,"word_overlap":overlap_pred}
# ROUTER (shipped): comparator on numeric/dim; per-stratum best-of(NLI, word-overlap) on prose/feature
router_pred={}
for b in BENCH:
    if b["stratum"] in NUMERIC_STRATA:
        router_pred[b["i"]]=int(bool(comparator(b["gold"], b["ctx_ids"])))
    else:
        router_pred[b["i"]]=PICK[best_nonnum[b["stratum"]]][b["i"]]
# GPU-FREE router variant: comparator on numeric/dim; word-overlap on prose/feature (no NLI, no GPU)
router_gf={}
for b in BENCH:
    router_gf[b["i"]]=int(bool(comparator(b["gold"], b["ctx_ids"]))) if b["stratum"] in NUMERIC_STRATA else overlap_pred[b["i"]]
rprint(f"[cyan]per-stratum best-of[/cyan] {best_nonnum}")
SC={"fuzzy":fuzzy_pred,"nli@0.5":nli05,"nli@0.8":nli08,"comparator":comp_pred,"word_overlap":overlap_pred,
    "router":router_pred,"router_gpufree":router_gf}
AG={k:agree(v) for k,v in SC.items()}
def fmt(x): return "  -  " if x is None else f"{x:5.2f}"
print(f"{'scorer':15} {'overall':>7} {'numeric':>8} {'dimension':>10} {'feature':>8} {'prose':>7}")
for k in ["fuzzy","nli@0.5","nli@0.8","comparator","word_overlap","router","router_gpufree"]:
    a=AG[k]; print(f"{k:15} {a['overall']:7.3f} {fmt(a['numeric']):>8} {fmt(a['dimension']):>10} {fmt(a['feature']):>8} {fmt(a['prose']):>7}")
# comparator on its home strata only (numeric+dim), excluding abstentions
comp_home=[b for b in BENCH if b["stratum"] in NUMERIC_STRATA]
comp_home_acc=np.mean([comp_pred[b["i"]]==b["label"] for b in comp_home])
rprint(f"\n[bold cyan]comparator home strata (numeric+dimension)[/bold cyan] {comp_home_acc:.3f} over {len(comp_home)} pairs "
       f"| abstains on {len(comp_abstain)} prose/feature golds")

In [ ]:
# NLI on the prose-fact stratum (registered >=85% clause) + disagreement forensics
prose=[b for b in BENCH if b["stratum"]=="prose"]
for thr,pred in [(0.5,nli05),(0.8,nli08)]:
    acc=np.mean([pred[b["i"]]==b["label"] for b in prose])
    rprint(f"[cyan]NLI@{thr}[/cyan] prose-fact stratum {acc:.3f} ({sum(pred[b['i']]==b['label'] for b in prose)}/{len(prose)})")
print("\n-- NLI value-identity failures (present numerics scored low / absent look-alikes scored high) --")
for b in BENCH:
    if b["stratum"] in ("numeric","dimension"):
        sc=nli_scores[b["i"]]
        if (b["label"]==1 and sc<0.5) or (b["label"]==0 and sc>=0.5):
            typ="present scored LOW" if b["label"]==1 else "absent scored HIGH"
            print(f"  #{b['i']:2} nli={sc:.3f} label={b['label']} [{typ}] {b['gold'][:26]!r}")
print("\n-- fuzzy false-positives on hard absences --")
for b in BENCH:
    if fuzzy_pred[b["i"]]==1 and b["label"]==0:
        print(f"  #{b['i']:2} [{b['stratum']}] {b['gold'][:26]!r} :: {b['rationale'][:52]}")

In [ ]:
# Machine-readable report
stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def clean(a): return {k:(None if v is None else float(v)) for k,v in a.items()}
best_single=max(["fuzzy","nli@0.5","nli@0.8","comparator","word_overlap"], key=lambda k:AG[k]["overall"])
report=dict(round="R18", hypothesis="H194", utc=stamp, graph="neo4j2", nli_model=NLI_MODEL, device=DEV,
  bench=dict(n=60, present=30, absent=30, strata={s:{"total":strata[s],"present":sp[(s,1)],"absent":sp[(s,0)]}
                                                    for s in ["numeric","dimension","feature","prose"]},
             locked="data/processed/instrument-bench-h194.json", divergence_from_runB=[9,19,21]),
  agreement_matrix={k:clean(AG[k]) for k in SC},
  comparator_home_strata=float(comp_home_acc), comparator_abstains=sorted(comp_abstain),
  nli_scores={str(b["i"]):float(nli_scores[b["i"]]) for b in BENCH},
  shipped_router=dict(rule="comparator for numeric/dimension golds; per-stratum best-of(NLI@0.5, word-overlap) for prose/feature",
                      per_stratum_choice=best_nonnum, overall=float(AG["router"]["overall"])),
  router_gpufree=dict(rule="comparator for numeric/dimension; word-overlap for prose/feature (no NLI, no GPU)",
                      overall=float(AG["router_gpufree"]["overall"])),
  best_single_scorer=dict(name=best_single, overall=float(AG[best_single]["overall"])),
  bars=dict(comparator_ge_90_overall=bool(AG["comparator"]["overall"]>=0.90),
            comparator_home_ge_90=bool(comp_home_acc>=0.90),
            comparator_wins_numeric=bool(AG["comparator"]["numeric"]>=max(AG["fuzzy"]["numeric"],AG["nli@0.5"]["numeric"],AG["nli@0.8"]["numeric"])),
            nli_prose_ge_85=bool(np.mean([nli05[b["i"]]==b["label"] for b in prose])>=0.85 or np.mean([nli08[b["i"]]==b["label"] for b in prose])>=0.85),
            router_beats_best_single=bool(AG["router"]["overall"]>=AG[best_single]["overall"])))
outp=Path("../reports")/f"instrument-bench-h194-{stamp}.json"; outp.write_text(json.dumps(report,indent=2,default=str))
rprint(f"[green]report[/green] -> {outp}")
rprint(f"[bold]bars[/bold] {json.dumps(report['bars'],indent=0)}")

## Verdict

The value comparator is the instrument the numeric strata needed; the router is the shipped form. See the agreement
matrix and bars above - the comparator wins the numeric and dimension strata outright and abstains cleanly on prose,
where NLI (or word overlap) carries the residual. The report captures the per-stratum matrix, the shipped router
decision, and each registered bar's PASS/FAIL. Honest note on scope: the comparator is a numeric specialist by design -
its standalone "overall" number is depressed by the prose/feature golds it abstains on; the router combines the two and
is the number that clears the registered overall bar.